# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {metadata.datePublished}\nVersion: {metadata.version}\nLicense: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will list all record sets in this dataset with their `@id`s, and for each, show its fields.

In [ ]:
# List all record sets and their fields (@id) in the dataset
record_sets = []
print("Record sets in the dataset:")
for rs in dataset.record_sets:
    print(f"  Record set name: {rs.name}")
    print(f"  @id: {rs.id}")
    fields = rs.fields
    print("  Fields:")
    for field in fields:
        print(f"    - {field.name} (@id: {field.id}) | dataType: {field.data_type}")
    print()
    record_sets.append(rs.id)

## 3. Data Extraction
Load data from each record set into a DataFrame. Fields are referenced by their `@id`.

In [ ]:
dataframes = {}

print('Extracting records for each record set...')
for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f'Record set {record_set_id}: {df.shape[0]} records, Columns: {df.columns.tolist()}')
        if df.shape[0] > 0:
            display(df.head())
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, and/or grouping data.

In [ ]:
# --- Pick the main protein data record set ---
# You may need to update the record set @id and field @id according to the overview above.

# For demonstration, pick the first record set with tabular protein data
if len(record_sets) == 0:
    print("No record sets available in the dataset.")
else:
    main_record_set = record_sets[0]
    df_main = dataframes.get(main_record_set, None)
    print(f'Using record set {main_record_set}.')

    if df_main is not None and len(df_main) > 0:
        print(f"Columns in the main dataframe:")
        print(df_main.columns.tolist())
        # Try to pick a numeric field for filtering/EDA
        # If MW (molecular weight) or similar field exists, use it; else pick the first float/integer field
        candidate_numeric_fields = [col for col in df_main.columns if df_main[col].dtype.kind in ("i", "u", "f")]
        if 'MW' in df_main.columns:
            numeric_field_id = 'MW'
        elif candidate_numeric_fields:
            numeric_field_id = candidate_numeric_fields[0]
        else:
            numeric_field_id = df_main.columns[0]  # fallback
        print(f"Selected numeric field: {numeric_field_id}")

        # Set a filtering threshold
        threshold = df_main[numeric_field_id].quantile(0.75) if pd.api.types.is_numeric_dtype(df_main[numeric_field_id]) else None
        if threshold is not None:
            filtered_df = df_main[df_main[numeric_field_id] > threshold]
            print(f"Filtered records with {numeric_field_id} > {threshold}:")
            display(filtered_df.head())

            # Normalize
            filtered_df[numeric_field_id + '_normalized'] = (
                filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
            ) / filtered_df[numeric_field_id].std()
            print(f"Normalized {numeric_field_id} for filtered records:")
            display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

            # Group by a categorical field (if available)
            group_field_id = None
            for cname in df_main.columns:
                if cname != numeric_field_id and df_main[cname].dtype == object:
                    group_field_id = cname
                    break
            if group_field_id:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
                print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
                display(grouped_df.head())
        else:
            print(f"Field {numeric_field_id} does not appear to be numeric!")
    else:
        print(f"No data loaded for record set {main_record_set}.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, we plot the distribution of the selected numeric field and, if available, a boxplot by a grouping field.

In [ ]:
# Simple visualization of distributions
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and filtered_df.shape[0] > 0:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id], bins=20, kde=True)
    plt.title(f'Distribution of {numeric_field_id} (Filtered)')
    plt.xlabel(numeric_field_id)
    plt.show()
    if group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f'{numeric_field_id} by {group_field_id} (Filtered)')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No filtered data available for plotting.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we demonstrated how to load and explore the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library. We overviewed record sets and their fields, extracted records using only Croissant schema `@id` references, performed basic EDA and visualized numeric distributions. For further analysis, refer to the dataset's Croissant schema and documentation for precise field semantics and full processing guidance.